# ASIRRA — configuration 6 sur T4

Exécuter les cellules dans l’ordre. L’entraînement conserve le modèle qui obtient le meilleur score en validation. La dernière cellule mesure son temps d’inférence sur les 2 048 images.

In [ ]:
%%bash

cd /content/
git clone https://github.com/Deyht/CIANNA -b experimental

cd CIANNA
cmake -B build -D USE_BLAS=off
cmake --build build
cd ..

In [ ]:
%%bash

cd /content

wget https://share.obspm.fr/s/6TBsCpAASeETH3S/download/asirra_bin_128.tar.gz
tar -xvzf asirra_bin_128.tar.gz

In [ ]:
!pip install wurlitzer # For proper display of CIANNA outputs in a notebook
%load_ext wurlitzer

import numpy as np
import matplotlib.pyplot as plt
import os, sys, gc, glob, time, cv2
from threading import Thread
from PIL import Image
import albumentations as A

sys.path.insert(0,glob.glob('/content/CIANNA/build/')[-1]) #if compiled with cmake
import CIANNA as cnn

In [ ]:
def data_prep(nb_images_per_iter, raw_image_size, image_size, test_mode=0):
  #Data arrays are declared as global so we can work in place to reduce RAM footprint
  global raw_images, input_data, targets, input_val, targets_val

  raw_images = np.reshape(np.fromfile("asirra_bin_%d.dat"%(raw_image_size), dtype="uint8"), (25000, raw_image_size, raw_image_size,3))

  if(test_mode == 0):
    input_data = np.zeros((nb_images_per_iter,3*image_size**2), dtype="float32") #CIANNA expects "float32" arrays
    targets = np.zeros((nb_images_per_iter,2), dtype="float32")

  input_val = np.zeros((2048,3*image_size**2), dtype="float32")
  targets_val = np.zeros((2048,2), dtype="float32")


def create_augmented_batch(A_transform):

  nb_images = np.shape(input_data)[0]

  for i in range(0,nb_images):

    l_class = np.random.randint(0,2)
    l_id = np.random.randint(0,12500 - 1024)
    #Last 1024 images of each class kept only for the val/test set

    patch = raw_images[l_class*12500+l_id]
    transformed = A_transform(image=patch)
    patch_aug = transformed['image']

    l_image_size = np.shape(patch_aug)[0]

    #CIANNA expects data formated as 2D numpy arrays representing a list of flattened images (with every channel flattened after the others)
    for depth in range(0,3): #We normalize based on mean pixel value
      input_data[i,depth*l_image_size**2:(depth+1)*l_image_size**2] = (patch_aug[:,:,depth].flatten("C") - 100.0)/155.0

    targets[i,:] = 0.05 #Label smoothing during training
    targets[i,l_class] = 0.95

  return input_data, targets


def create_validation_set(A_transform):

  for i in range(0,2048):

    l_class = i // 1024

    patch = raw_images[(1+l_class)*(12500 - 1024) + i]
    transformed = A_transform(image=patch)
    patch_aug = transformed['image']

    l_image_size = np.shape(patch_aug)[0]

    for depth in range(0,3):
      input_val[i,depth*l_image_size**2:(depth+1)*l_image_size**2] = (patch_aug[:,:,depth].flatten("C") - 100.0)/155.0

    targets_val[i,:] = 0.0
    targets_val[i,l_class] = 1.0

  return input_val, targets_val


def free_data_helper():
  global raw_images, input_data, targets, input_val, targets_val
  if 'raw_images' in globals():
    del (raw_images)
  if 'input_data' in globals():
    del (input_data)
  if 'targets' in globals():
    del (targets)
  if 'input_val' in globals():
    del (input_val)
  if 'targets_val' in globals():
    del (targets_val)
  gc.collect()
  return

## Entraînement

In [ ]:
import torch
import csv
from pathlib import Path


raw_image_size = 128
image_size = 128  # Config 6 : exploiter toute la résolution des images ASIRRA.
eval_batch = 128
precision = "FP16C_FP32A"
train_precision = "FP32C_FP32A"
val_transform = A.Compose([A.Resize(image_size, image_size, interpolation=1)])

def build_model():
  relu = cnn.relu(leaking=0.05, saturation=64.0)
  cnn.input(spatial_shape=[image_size, image_size, 1], nb_features=3)
  cnn.conv(f_size=(4,4), stride=(2,2), padding=(1,1), nb_filters=24, activation=relu)
  for width in (48, 96):
    cnn.pool(p_size=(2,2), p_type="MAX")
    for _ in range(2):
      cnn.conv(f_size=(3,3), padding=(1,1), nb_filters=width, activation=relu)
  cnn.pool(p_size=(2,2), p_type="MAX")
  cnn.conv(f_size=(3,3), padding=(1,1), nb_filters=128, activation=relu)
  cnn.conv(f_size=(1,1), nb_filters=2, activation="LIN")
  cnn.pool(p_size=(1,1), p_global=1, p_type="AVG", activation="SMAX")

nb_images_per_iter = 2048
train_transform = A.Compose([
  A.Resize(image_size, image_size, interpolation=1),
  A.ColorJitter(brightness=(0.85,1.15), contrast=(0.85,1.15), saturation=(0.85,1.15), hue=0.05, p=0.8),
  A.ToGray(p=0.02), A.HorizontalFlip(p=0.5),
  A.Affine(scale=(0.9,1.1), rotate=(-10,10), interpolation=1, p=0.3),
])

def data_augm():
  input_data, targets = create_augmented_batch(train_transform)
  cnn.delete_dataset("TRAIN_buf", silent=1)
  cnn.create_dataset("TRAIN_buf", nb_images_per_iter, input_data[:,:], targets[:,:], silent=1)
  return

cnn.init(b_size=32, optimizer=cnn.rms_prop(momentum=0.9), wema=0, dynamic_load=1,
  comp_meth='C_CUDA', mixed_precision=train_precision)
build_model()

data_prep(nb_images_per_iter, raw_image_size, image_size)

input_val, targets_val = create_validation_set(val_transform)
cnn.create_dataset("VALID", 2048, input_val, targets_val)
cnn.create_dataset("TEST", 2048, input_val, targets_val)


input_data, targets = create_augmented_batch(train_transform)
cnn.create_dataset("TRAIN", nb_images_per_iter, input_data[:,:], targets[:,:])

warmup = 10
cycle_length = 500
l_r_max = 3e-4
l_r_min = 3e-6

run_dir = Path("/content/config6")
run_dir.mkdir(exist_ok=True)
checkpoint = run_dir / "config6_best.dat"
log_path = run_dir / "training.csv"
best_accuracy, best_iter = -1.0, 0
torch.cuda.synchronize()
train_start = time.perf_counter()
with log_path.open("w", newline="") as log:
  writer = csv.writer(log)
  writer.writerow(["iteration", "images_seen", "training_seconds", "validation_accuracy_pct", "best_iteration"])
  for i in range(cycle_length):

    t = Thread(target=data_augm)
    t.start()

    if(i < warmup):
      l_r = l_r_min + (l_r_max - l_r_min) * min(i/warmup,1.0)
    else: # Cyclical cosin learning rate decay with warm restart
      in_cycle_progress = (i%cycle_length)/cycle_length
      l_r = l_r_min + 0.5*(l_r_max - l_r_min)*(1 + np.cos(np.pi*in_cycle_progress))

    cnn.train(nb_iter=1, learning_rate=l_r, weight_decay=0.01, decoupled_wdecay=1,
      confmat=1, control_interv=50, save_every=0, save_optim_every=0, save_bin=1, TC_scale_factor=1.0, shuffle_every=0)

    if(i == 0):
      cnn.perf_eval()

    t.join()
    cnn.swap_data_buffers("TRAIN")
    if (i + 1) % 50 == 0:
      preds = cnn.forward(drop_mode="AVG_MODEL", no_error=1, saving=0, return_output=1)
      accuracy = float(np.mean(preds[:, :2].argmax(1) == targets_val.argmax(1)))
      if accuracy > best_accuracy:
        best_accuracy, best_iter = accuracy, i + 1
        cnn.save(str(checkpoint), bin=1)
      torch.cuda.synchronize()
      elapsed = time.perf_counter() - train_start
      writer.writerow([i + 1, (i + 1) * nb_images_per_iter, round(elapsed, 2),
                       round(100 * accuracy, 4), best_iter])
      log.flush()
      print(f"Iter {i+1} | {elapsed/60:.2f} min | accuracy {accuracy:.2%} | meilleur checkpoint {best_iter}", flush=True)
print(f"Temps total d’entraînement : {elapsed/60:.2f} min. Log : {log_path}")
free_data_helper()

## Évaluation sur les 2 048 images

In [ ]:
import torch
val_transform = A.Compose([
    A.Resize(height=image_size, width=image_size, interpolation=1),
])

cnn.init(b_size=eval_batch, inference_only=1, dynamic_load=1,
  comp_meth='C_CUDA', mixed_precision="FP16C_FP32A")

data_prep(0, raw_image_size, image_size, test_mode=1)

input_test, targets_test = create_validation_set(val_transform)
cnn.create_dataset("TEST", 2048, input_test[:,:], targets_test[:,:])


cnn.input(spatial_shape=[image_size,image_size,1], nb_features=3)
cnn.load(str(checkpoint), best_iter, nb_skip_layers=1, bin=1, silent=1)
pred = cnn.forward(drop_mode="AVG_MODEL", no_error=1, saving=0, return_output=1)
torch.cuda.synchronize()
times = []
for _ in range(5):
  start = time.perf_counter()
  cnn.forward(drop_mode="AVG_MODEL", no_error=1, saving=0, return_output=0)
  torch.cuda.synchronize()
  times.append(1000 * (time.perf_counter() - start))

median_ms = np.median(times)
accuracy = np.mean(pred[:, :2].argmax(axis=1) == targets_test.argmax(axis=1))
print(f"2 048 images : {np.round(times, 1)} ms | médiane : {median_ms:.1f} ms | budget < 300 ms : {median_ms < 300}")
print(f"Accuracy : {100 * accuracy:.2f} %")
free_data_helper()